# Homework 4 - Computing Point-in-Time Residual Returns
In this homework, we will use regressions to compute beta-adjusted "residual" returns in a point-in-time fashion suitable for backtesting / live trading.


1. Download Daily Bars for FB, AAPL, AMZN, NFLX, GOOGL and QQQ from yahoo finance starting 2016-01-01. Use the Adj Close to compute daily returns.



5. Compare the pairwise correlations of the residual returns to that of the original returns. What do you notice?
6. Compute the information ratio for each of these stocks and compare that to the sharpe ratio.


In [1]:
import yfinance
import pandas as pd
import numpy as np
import matplotlib as plt
import statsmodels.api as sm

In [2]:
stocks = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL']
benchmark = ['QQQ']


In [ ]:
# to download and store the data in a pickle file.
univ = stocks + benchmark
pxs = yfinance.download(univ, start='2016-01-01')['Close']
pxs.to_pickle('homework4_prices.pkl')

In [3]:
pxs = pd.read_pickle('homework4_prices.pkl')

In [4]:
# 1. Compute daily returns
ret = pxs / pxs.shift(1) - 1

2. Now, let's compute the beta of FB, AAPL, AMZN, NFLX, GOOGL using QQQ as our benchmark. You can think of this as the beta these stocks have to their industry (tech). In practice,  we have to use some lookback window to compute the beta. Let's use 252 (1 year, excluding wknds/holidays). So, for each day, the betas should be computed using the most recent 252 data points.

In [5]:
corr = ret.rolling(252).corr(ret['QQQ'])
vol = ret.rolling(252).std()
beta = (corr * vol).divide(vol['QQQ'], axis=0)

3. Using the betas, compute an "alpha" on each day. This is also known as a "residual return".

In [6]:
alpha = ret - beta.mul(ret['QQQ'],0)

4. Compare the volatility of the residual returns to that of the original returns. What do you notice?

In [8]:
vol = {}
vol['orig'] = ret.std()*np.sqrt(252)
vol['resid'] = alpha.std()*np.sqrt(252)
vol = pd.DataFrame(vol).drop('QQQ')
vol

,orig,resid
Ticker,,
AAPL,0.290086,0.171891
AMZN,0.327885,0.205446
GOOGL,0.287279,0.182837
META,0.385081,0.277969
NFLX,0.420044,0.330640


In [9]:
ret.std() * np.sqrt(252)

Ticker
AAPL     0.290086
AMZN     0.327885
GOOGL    0.287279
META     0.385081
NFLX     0.420044
QQQ      0.222522
dtype: float64

In [10]:
alpha.std() * np.sqrt(252)

Ticker
AAPL     1.718911e-01
AMZN     2.054464e-01
GOOGL    1.828366e-01
META     2.779694e-01
NFLX     3.306395e-01
QQQ      6.236360e-16
dtype: float64

In [17]:
ret.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,0.568313,0.605310,0.516900,0.422939,0.801649
AMZN,0.568313,1.000000,0.634944,0.604728,0.519373,0.764759
GOOGL,0.605310,0.634944,1.000000,0.608922,0.437536,0.782346
META,0.516900,0.604728,0.608922,1.000000,0.451676,0.697355
NFLX,0.422939,0.519373,0.437536,0.451676,1.000000,0.580782
QQQ,0.801649,0.764759,0.782346,0.697355,0.580782,1.000000


In [18]:
alpha.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,-0.094133,-0.055674,-0.081770,-0.095311,-0.019595
AMZN,-0.094133,1.000000,0.070587,0.127190,0.132903,-0.000131
GOOGL,-0.055674,0.070587,1.000000,0.126967,-0.059554,-0.018715
META,-0.081770,0.127190,0.126967,1.000000,0.078763,-0.005618
NFLX,-0.095311,0.132903,-0.059554,0.078763,1.000000,-0.014475
QQQ,-0.019595,-0.000131,-0.018715,-0.005618,-0.014475,1.000000
